# Generación de dataframes para comparar
En el dataset existente se observó que para un mismo objeto estelar, pueden haber más de una medición y que, en algunos casos, incluso puede suceder que la clase en el target difiera. En función de esto se decidió quenerar dos datasets distintos para comparar y tratar de entender el motivo detrás de esta situación:

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../dataset/star_classification.csv')
# Cantidad de registros anómalos
anomalias = df[(df['u'] == -9999) | (df['g'] == -9999) | (df['z'] == -9999)]

df = df[(df['u'] != -9999) & (df['g'] != -9999) & (df['z'] != -9999)]

In [3]:
# ============================================================
# Generación del DataFrame 1: Clase mayoritaria por obj_ID
# ============================================================

# 1. Identificar grupos con más de un registro por obj_ID
group_counts = df.groupby('obj_ID').size()
duplicated_obj_ids = group_counts[group_counts > 1].index
unique_obj_ids = group_counts[group_counts == 1].index

print(f"Total de obj_IDs únicos: {df['obj_ID'].nunique()}")
print(f"obj_IDs con un solo registro: {len(unique_obj_ids)}")
print(f"obj_IDs con múltiples registros: {len(duplicated_obj_ids)}")
print(f"Total de registros en el dataset original: {len(df)}")
print("=" * 70)

# 2. Analizar la situación de cada grupo con múltiples registros
rows_to_keep = []
total_discarded = 0

for obj_id in duplicated_obj_ids:
    group = df[df['obj_ID'] == obj_id]
    class_counts = group['class'].value_counts()
    
    # Verificar si hay más de una clase en el grupo
    has_conflict = len(class_counts) > 1
    majority_class = class_counts.idxmax()
    majority_count = class_counts.max()
    total_in_group = len(group)
    
    if has_conflict:
        minority_count = total_in_group - majority_count
        total_discarded += minority_count
        # print(f"\n🔀 obj_ID: {obj_id}")
        # print(f"   Total registros: {total_in_group}")
        # print(f"   Distribución de clases: {dict(class_counts)}")
        # print(f"   Clase mayoritaria: '{majority_class}' ({majority_count} registros)")
        # print(f"   Registros descartados (clases minoritarias): {minority_count}")
    
    # Conservar solo los registros de la clase mayoritaria
    rows_to_keep.append(group[group['class'] == majority_class])

# 3. Agregar los registros de obj_IDs únicos (sin duplicados)
unique_rows = df[df['obj_ID'].isin(unique_obj_ids)]

# 4. Construir el DataFrame filtrado
df_majority = pd.concat([unique_rows] + rows_to_keep, ignore_index=True)

print("\n" + "=" * 70)
print("RESUMEN FINAL")
print("=" * 70)
print(f"Registros originales: {len(df)}")
print(f"Registros descartados por clase minoritaria: {total_discarded}")
print(f"Registros en df_majority: {len(df_majority)}")
print(f"\nDistribución de clases en df original:")
print(df['class'].value_counts())
print(f"\nDistribución de clases en df_majority:")
print(df_majority['class'].value_counts())


Total de obj_IDs únicos: 78052
obj_IDs con un solo registro: 62693
obj_IDs con múltiples registros: 15359
Total de registros en el dataset original: 99999

RESUMEN FINAL
Registros originales: 99999
Registros descartados por clase minoritaria: 6340
Registros en df_majority: 93659

Distribución de clases en df original:
class
GALAXY    59445
STAR      21593
QSO       18961
Name: count, dtype: int64

Distribución de clases en df_majority:
class
GALAXY    57928
STAR      19345
QSO       16386
Name: count, dtype: int64
